In [1]:
import os
print(os.getcwd())

/Users/mme.nouko/kayak_project/src


In [2]:
import requests
import pandas as pd
import time
import plotly.express as px

In [3]:
# --- Vérification préalable (à garder ou supprimer après validation) ---
print("Dossier courant :", os.getcwd())

# --- Liste complète et exacte des 35 villes imposées par le projet ---
cities = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen",
    "Paris", "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg",
    "Colmar", "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble",
    "Lyon", "Gorges du Verdon", "Bormes les Mimosas", "Cassis",
    "Marseille", "Aix en Provence", "Avignon", "Uzes", "Nimes",
    "Aigues Mortes", "Saintes Maries de la mer", "Collioure",
    "Carcassonne", "Ariege", "Toulouse", "Montauban", "Biarritz",
    "Bayonne", "La Rochelle"
]

def get_coordinates(city_name):
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": city_name, "format": "json", "limit": 1}
    headers = {"User-Agent": "KayakProjectJedha/1.0"}

    response = requests.get(url, params=params, headers=headers)

    if response.status_code == 200:
        data = response.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    return None, None

results = []
for city in cities:
    lat, lon = get_coordinates(city)
    results.append({"city": city, "lat": lat, "lon": lon})
    time.sleep(1)

df_coords = pd.DataFrame(results)
df_coords["id"] = df_coords.index

# Vérification avant sauvegarde : on doit avoir 35 lignes, sans None
print(f"Nombre de villes récupérées : {len(df_coords)}")
print(f"Villes sans coordonnées : {df_coords['lat'].isna().sum()}")

df_coords.to_csv("../csv/cities_coordinates.csv", index=False)
print(df_coords)

Dossier courant : /Users/mme.nouko/kayak_project/src
Nombre de villes récupérées : 35
Villes sans coordonnées : 0
                            city        lat       lon  id
0              Mont Saint Michel  48.635954 -1.511460   0
1                        St Malo  48.649518 -2.026041   1
2                         Bayeux  49.276462 -0.702474   2
3                       Le Havre  49.493898  0.107973   3
4                          Rouen  49.440459  1.093966   4
5                          Paris  48.853495  2.348391   5
6                         Amiens  49.894171  2.295695   6
7                          Lille  50.636565  3.063528   7
8                     Strasbourg  48.584614  7.750713   8
9   Chateau du Haut Koenigsbourg  48.249382  7.343941   9
10                        Colmar  48.077752  7.357964  10
11                     Eguisheim  48.044797  7.307962  11
12                      Besancon  47.238022  6.024362  12
13                         Dijon  47.321581  5.041470  13
14              

In [4]:
import os
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "src" else NOTEBOOK_DIR
CSV_DIR = PROJECT_ROOT / "csv"

df_coords = pd.read_csv(CSV_DIR / "cities_coordinates.csv")

def get_weather(lat, lon):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {"lat": lat, "lon": lon, "units": "metric", "appid": API_KEY}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    print(f"Erreur {response.status_code} pour lat={lat}, lon={lon}")
    return None

weather_results = []
for _, row in df_coords.iterrows():
    data = get_weather(row["lat"], row["lon"])
    if data and "list" in data:
        forecasts = data["list"]  # 40 points (5 jours x 8 tranches de 3h)
        temps = [f["main"]["temp"] for f in forecasts]
        humidities = [f["main"]["humidity"] for f in forecasts]
        pop_values = [f.get("pop", 0) for f in forecasts]
        rain_values = [f.get("rain", {}).get("3h", 0) for f in forecasts]

        weather_results.append({
            "id": row["id"],
            "city": row["city"],
            "avg_temp": sum(temps) / len(temps),
            "avg_humidity": sum(humidities) / len(humidities),
            "total_pop": sum(pop_values),
            "total_rain": sum(rain_values)
        })
    time.sleep(0.5)

df_weather = pd.DataFrame(weather_results)
df_weather.to_csv(CSV_DIR / "cities_weather.csv", index=False)
print(df_weather)

    id                          city  avg_temp  avg_humidity  total_pop  \
0    0             Mont Saint Michel  17.40000        72.650       6.90   
1    1                       St Malo  17.74100        72.800       6.21   
2    2                        Bayeux  16.99500        72.600       4.44   
3    3                      Le Havre  18.13550        71.600       9.05   
4    4                         Rouen  17.65875        69.675       1.19   
5    5                         Paris  20.89575        50.150       4.09   
6    6                        Amiens  17.64825        66.250       2.42   
7    7                         Lille  17.78650        62.100       2.01   
8    8                    Strasbourg  20.95700        56.350       3.50   
9    9  Chateau du Haut Koenigsbourg  18.41025        61.450       2.71   
10  10                        Colmar  21.10925        61.225       3.07   
11  11                     Eguisheim  20.85025        61.200       2.94   
12  12                   

In [5]:
import pandas as pd
from pathlib import Path
import os

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "src" else NOTEBOOK_DIR
CSV_DIR = PROJECT_ROOT / "csv"

df_coords = pd.read_csv(CSV_DIR / "cities_coordinates.csv")
df_weather = pd.read_csv(CSV_DIR / "cities_weather.csv")

df_full = df_coords.merge(df_weather, on=["id", "city"], how="inner")

df_full["weather_score"] = df_full["total_rain"] + (df_full["total_pop"] * 10)

df_top5 = df_full.sort_values("weather_score", ascending=True).head(5)

df_top5.to_csv(CSV_DIR / "top5_destinations.csv", index=False)
print(df_top5[["city", "avg_temp", "avg_humidity", "total_pop", "total_rain", "weather_score"]])

                        city  avg_temp  avg_humidity  total_pop  total_rain  \
19                    Cassis  25.37375        62.300       1.02        0.39   
26  Saintes Maries de la mer  24.61100        70.775       1.07        0.61   
4                      Rouen  17.65875        69.675       1.19        1.14   
25             Aigues Mortes  25.29700        66.175       1.23        1.07   
20                 Marseille  26.09950        62.025       1.87        1.25   

    weather_score  
19          10.59  
26          11.31  
4           13.04  
25          13.37  
20          19.95  


In [6]:
import plotly.express as px

fig = px.scatter_mapbox(
    df_top5,
    lat="lat",
    lon="lon",
    hover_name="city",
    hover_data={"avg_temp": True, "total_rain": True, "lat": False, "lon": False},
    color="weather_score",
    color_continuous_scale="Blues_r",
    size=[15] * len(df_top5),
    zoom=4.5,
    center={"lat": 46.5, "lon": 2.5},
    mapbox_style="open-street-map",
    title="Top 5 destinations - Meilleure météo sur 5 jours"
)

fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

/var/folders/d2/2h9xzbcd25x76wt_vzczswgm0000gn/T/ipykernel_1829/1905025517.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [7]:
import scrapy
from scrapy.crawler import CrawlerProcess

class BookingSpider(scrapy.Spider):
    name = "booking_spider"

    custom_settings = {
        "FEEDS": {"csv/hotels_raw.json": {"format": "json"}},
        "AUTOTHROTTLE_ENABLED": True,
        "AUTOTHROTTLE_START_DELAY": 2,
        "USER_AGENT": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "LOG_LEVEL": "INFO"
    }

    def __init__(self, cities=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.start_urls = [
            f"https://www.booking.com/searchresults.fr.html?ss={city.replace(' ', '+')}"
            for city in cities
        ]

    def parse(self, response):
        city = response.url.split("ss=")[1].split("&")[0].replace("+", " ")
        hotels = response.xpath('//div[@data-testid="property-card"]')

        for hotel in hotels:
            yield {
                "city": city,
                "name": hotel.xpath('.//div[@data-testid="title"]/text()').get(),
                "url": hotel.xpath('.//a[@data-testid="title-link"]/@href').get(),
                "rating": hotel.xpath('.//div[@data-testid="review-score"]//text()').get(),
                "description": hotel.xpath('.//div[@data-testid="recommended-units"]//text()').get(),
            }

In [8]:
import pandas as pd
import json
import os
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "src" else NOTEBOOK_DIR
CSV_DIR = PROJECT_ROOT / "csv"

print("Dossier notebook :", NOTEBOOK_DIR)
print("Dossier csv utilisé :", CSV_DIR)

df_top5 = pd.read_csv(CSV_DIR / "top5_destinations.csv")

with open(CSV_DIR / "hotels_raw.json", "r", encoding="utf-8") as f:
    hotels_data = json.load(f)
df_hotels = pd.DataFrame(hotels_data)

df_final = df_hotels.merge(
    df_top5[["id", "city", "avg_temp", "avg_humidity", "total_rain"]],
    on="city",
    how="left"
)

df_final.to_csv(CSV_DIR / "kayak_final_dataset.csv", index=False)
print(f"Dataset final : {len(df_final)} hôtels, {df_final['city'].nunique()} villes")
print(df_final.head())

Dossier notebook : /Users/mme.nouko/kayak_project/src
Dossier csv utilisé : /Users/mme.nouko/kayak_project/csv
Dataset final : 125 hôtels, 5 villes
        city                                               name  \
0  Collioure                         Hôtel Princes de Catalogne   
1  Collioure                   Charmant Studio près de la Plage   
2  Collioure  Logement 50 m de la plage Arrivée et Départ Au...   
3  Collioure       Appartement Centre COLLIOURE CLIM PATIO WIFI   
4  Collioure  Residence Pierre & Vacances Les Balcons de Col...   

                                                 url                rating  \
0  https://www.booking.com/hotel/fr/princes-de-ca...  Avec une note de 8,4   
1  https://www.booking.com/hotel/fr/charmant-stud...  Avec une note de 8,0   
2  https://www.booking.com/hotel/fr/logement-en-b...  Avec une note de 8,8   
3  https://www.booking.com/hotel/fr/appartement-c...  Avec une note de 9,3   
4  https://www.booking.com/hotel/fr/maevabalconsd...  Avec 

In [9]:
# 1. Nettoyage de la colonne 'rating' (extraction des chiffres et conversion en float)
# Transforme "Avec une note de 8,4" en 8.4
df_final['score'] = df_final['rating'].str.extract(r'(\d+,\d+)')[0].str.replace(',', '.').astype(float)

In [10]:
# 2. Ajout des coordonnées (lat/lon) depuis df_coords pour l'affichage sur la carte
df_final = df_final.merge(df_coords[['city', 'lat', 'lon']], on='city', how='left')

In [11]:
# 3. Sélection des 20 meilleurs hôtels (triés par le nouveau score)
top_20_hotels = df_final.sort_values(by='score', ascending=False).head(20)

In [12]:
# 4. Génération de la carte interactive
fig_hotels = px.scatter_mapbox(
    top_20_hotels, 
    lat="lat", 
    lon="lon", 
    hover_name="name", 
    hover_data=["city", "score"],
    color="score",
    color_continuous_scale="Viridis",
    size_max=15,
    zoom=5, 
    height=500,
    mapbox_style="open-street-map",
    title="Top 20 des meilleurs hôtels (Note Booking)"
)
fig_hotels.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_hotels.show()

/var/folders/d2/2h9xzbcd25x76wt_vzczswgm0000gn/T/ipykernel_1829/3628230018.py:2: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_hotels = px.scatter_mapbox(


In [ ]:
import boto3
import pandas as pd
from botocore.exceptions import ClientError

# Mes identifiants AWS
# /!\ ATTENTION : Ne jamais laisser ces clés dans mon code final sur GitHub !
ACCESS_KEY = "XXX"
SECRET_KEY = "XXX"

session = boto3.Session(
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
)

s3 = session.resource("s3")
bucket_name = "kayak-datalake-jedha-carelle-2026" 

# Création de mon Data Lake (avec gestion de l'erreur s'il existe déjà)
try:
    print(f"Création du bucket '{bucket_name}'...")
    s3.create_bucket(
        Bucket=bucket_name, 
        CreateBucketConfiguration={'LocationConstraint': 'eu-west-3'}
    )
except ClientError as e:
    if e.response['Error']['Code'] in ['BucketAlreadyOwnedByYou', 'BucketAlreadyExists']:
        print("Le bucket existe déjà sur AWS, on passe à la suite !")
    else:
        raise e

# Envoi de mon fichier CSV final dans le Bucket S3
print("Envoi du dataset vers AWS S3...")
s3.Bucket(bucket_name).upload_file("../csv/kayak_final_dataset.csv", "kayak_final_dataset.csv")

print("Étape 3 validée : Fichier envoyé avec succès sur Amazon S3 !")

Création du bucket 'kayak-datalake-jedha-carelle-2026'...
Le bucket existe déjà sur AWS, on passe à la suite !
Envoi du dataset vers AWS S3...
Étape 3 validée : Fichier envoyé avec succès sur Amazon S3 !


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Paramètres de connexion AWS RDS (Corrigés avec ton vrai endpoint)
USER = "postgres"
PASSWORD = "XXX_MOT_DE_PASSE_CACHE_XXX"
ENDPOINT = "database-1.cnk88ao8o695.eu-north-1.rds.amazonaws.com"
PORT = "5432"
DB_NAME = "postgres"

# 2. Création de la connexion SQLAlchemy
engine_url = f"postgresql://{USER}:{PASSWORD}@{ENDPOINT}:{PORT}/{DB_NAME}"
engine = create_engine(engine_url, echo=False)

# 3. Chargement du CSV
df_final = pd.read_csv("../csv/kayak_final_dataset.csv")

# Nettoyage rapide du score si nécessaire
if df_final['rating'].dtype == 'object':
    df_final['score'] = df_final['rating'].str.extract(r'(\d+,\d+)')[0].str.replace(',', '.').astype(float)
else:
    df_final['score'] = df_final['rating']

# 4. Envoi des données dans AWS RDS
print("Connexion et transfert vers AWS RDS en cours...")
df_final.to_sql("destinations_hotels", con=engine, if_exists="replace", index=False)

print("Étape 4 validée avec succès : Données stockées dans AWS RDS !")

Connexion et transfert vers AWS RDS en cours...
Étape 4 validée avec succès : Données stockées dans AWS RDS !


# CARTE 1 : TOP 5 DES DESTINATIONS

In [15]:
import plotly.express as px
import pandas as pd

# On recharge le dataset proprement pour être sûr qu'il est tout neuf en mémoire
df_final = pd.read_csv("../csv/kayak_final_dataset.csv")

# Dictionnaire des coordonnées GPS (SANS espaces invisibles)
coords_villes = {
    'Bordeaux': {'lat': 44.8378, 'lon': -0.5792},
    'Marseille': {'lat': 43.2965, 'lon': 5.3698},
    'Paris': {'lat': 48.8566, 'lon': 2.3522},
    'Lyon': {'lat': 45.7640, 'lon': 4.8357},
    'Nice': {'lat': 43.7102, 'lon': 7.2620},
    'Nantes': {'lat': 47.2184, 'lon': -1.5536},
    'Strasbourg': {'lat': 48.5734, 'lon': 7.7521},
    'Montpellier': {'lat': 43.6108, 'lon': 3.8767}
}

# Injection propre des coordonnées
df_final['lat'] = df_final['city'].map(lambda x: coords_villes.get(x, {'lat': 46.6033})['lat'])
df_final['lon'] = df_final['city'].map(lambda x: coords_villes.get(x, {'lon': 1.8883})['lon'])

# CARTE 2 : TOP 20 DES HÔTELS

In [16]:
df_cities = df_final.drop_duplicates(subset=['city']).head(5) 

fig_dest = px.scatter_mapbox(
    df_cities, 
    lat="lat", 
    lon="lon", 
    hover_name="city", 
    color_discrete_sequence=["red"],
    size_max=15,
    zoom=4, 
    height=450,
    mapbox_style="open-street-map", 
    title="Top 5 des meilleures destinations"
)
fig_dest.show()

# --- CARTE 2 : TOP 20 DES HÔTELS ---
# Nettoyage du score pour le classement
if df_final['rating'].dtype == 'object':
    df_final['score'] = df_final['rating'].str.extract(r'(\d+,\d+)')[0].str.replace(',', '.').astype(float)
else:
    df_final['score'] = df_final['rating']

top_20_hotels = df_final.sort_values(by='score', ascending=False).head(20)

fig_hotels = px.scatter_mapbox(
    top_20_hotels, 
    lat="lat", 
    lon="lon", 
    hover_name="name", 
    hover_data=["city", "score"],
    color="score",
    color_continuous_scale="Viridis",
    zoom=5, 
    height=550,
    mapbox_style="open-street-map", 
    title="Top 20 des meilleurs hôtels (Note Booking)"
)
fig_hotels.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})

/var/folders/d2/2h9xzbcd25x76wt_vzczswgm0000gn/T/ipykernel_1829/3869234052.py:3: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_dest = px.scatter_mapbox(


/var/folders/d2/2h9xzbcd25x76wt_vzczswgm0000gn/T/ipykernel_1829/3869234052.py:26: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_hotels = px.scatter_mapbox(
